In [ ]:
import torch
import torchvision.transforms as transforms
from sklearn.metrics import roc_auc_score
import pandas as pd
from PIL import Image
import glob
import numpy as np

# Define label columns
label_cols = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 
          'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 
          'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
          'GENDER_Female', 'GENDER_Male',
          'PRIMARY_RACE_Asian', 'PRIMARY_RACE_Asian - Historical Conv', 'PRIMARY_RACE_Asian, Hispanic',
          'PRIMARY_RACE_Asian, non-Hispanic', 'PRIMARY_RACE_Black or African American', 'PRIMARY_RACE_Black, Hispanic', 'PRIMARY_RACE_Black, non-Hispanic',
          'PRIMARY_RACE_White', 'PRIMARY_RACE_White or Caucasian', 'PRIMARY_RACE_White, Hispanic', 'PRIMARY_RACE_White, non-Hispanic',
          'AGE_GROUP_AGE_0_30', 'AGE_GROUP_AGE_31_50', 'AGE_GROUP_AGE_51_70', 'AGE_GROUP_AGE_71_plus']

# Set up image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Assuming ImageNet normalization
])
import torch
import torchvision.models as models  # Adjust this to your model's library

# Step 1: Define the model architecture
model = models.densenet121(pretrained=False, num_classes=31)  # Change num_classes as per your setup

# Step 2: Load the state dictionary
model.load_state_dict(torch.load("/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/densnet12_real_vs_real_training/model_epoch_9.pth"))

# Step 3: Switch to evaluation mode
model.eval()

print("Model loaded and ready for inference!")


model.eval()

# Load generated images
image_paths = glob.glob('/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/LORA_fine_tuning_stable_diffusion_model/rough_gen_for_image_checking*.jpg')

# Store predictions and true labels
all_preds = []
all_labels = []

# Loop through images and predict
for image_path in image_paths:
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0)  # Add batch dimension
    
    with torch.no_grad():
        logits = model(image)
        probs = torch.sigmoid(logits).cpu().numpy().flatten()

    # Assuming you have ground truth labels stored in a CSV file
    filename = image_path.split('/')[-1]
    true_labels = pd.read_csv('path_to_labels.csv')
    true_labels = true_labels[true_labels['filename'] == filename][label_cols].values.flatten()

    all_preds.append(probs)
    all_labels.append(true_labels)

# Convert to numpy arrays and ensure they are 2D
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Reshape if they are 1D
if all_preds.ndim == 1:
    all_preds = all_preds.reshape(-1, len(label_cols))
if all_labels.ndim == 1:
    all_labels = all_labels.reshape(-1, len(label_cols))

# Calculate AUC for each label
auc_scores = {}
for i, label in enumerate(label_cols):
    try:
        # Check if both classes are present
        if len(np.unique(all_labels[:, i])) > 1:
            auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
            auc_scores[label] = auc
            print(f'AUC for {label}: {auc:.4f}')
        else:
            print(f"Skipping AUC for {label} - only one class present.")
    except ValueError as e:
        print(f"AUC calculation failed for {label}: {e}")

print("\nAUC Scores for All Labels:")
for label, auc in auc_scores.items():
    print(f"{label}: {auc:.4f}")


In [1]:
import torch
import torchvision.transforms as transforms
import glob
import numpy as np
import os
from PIL import Image

# Define label columns for diseases and demographics (age, gender, race)
label_cols = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 
              'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 
              'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices',
              'GENDER_Female', 'GENDER_Male',
              'PRIMARY_RACE_Asian', 'PRIMARY_RACE_Asian - Historical Conv', 'PRIMARY_RACE_Asian, Hispanic',
              'PRIMARY_RACE_Asian, non-Hispanic', 'PRIMARY_RACE_Black or African American', 'PRIMARY_RACE_Black, Hispanic', 'PRIMARY_RACE_Black, non-Hispanic',
              'PRIMARY_RACE_White', 'PRIMARY_RACE_White or Caucasian', 'PRIMARY_RACE_White, Hispanic', 'PRIMARY_RACE_White, non-Hispanic',
              'AGE_GROUP_AGE_0_30', 'AGE_GROUP_AGE_31_50', 'AGE_GROUP_AGE_51_70', 'AGE_GROUP_AGE_71_plus']

# Set up image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Assuming ImageNet normalization
])

# Import necessary model library
import torch
import torchvision.models as models  # Adjust this to your model's library

# Step 1: Define the model architecture
model = models.densenet121(pretrained=False, num_classes=31)  # Change num_classes as per your setup

# Step 2: Load the state dictionary
try:
    model.load_state_dict(torch.load("/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/densnet12_real_vs_real_training/model_epoch_9.pth"))
    print("Model weights loaded successfully.")
except Exception as e:
    print(f"Error loading model weights: {e}")
    exit()

# Step 3: Switch to evaluation mode
model.eval()
print("Model loaded and ready for inference!")

# Set the directory containing the generated images
image_folder = '/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/LORA_fine_tuning_stable_diffusion_model/rough_gen_for_image_checking/'

# List all image files in the folder (handles both .jpg and .png)
image_paths = glob.glob(os.path.join(image_folder, '*.[jp][np][g]'))

# Ensure there are images in the folder
if not image_paths:
    print("No images found in the specified directory!")
    exit()

print(f"Found {len(image_paths)} images.")

# Set a threshold for binary classification (e.g., 0.5)
threshold = 0.5

# Loop through all images in the folder
for image_path in image_paths:
    try:
        # Debugging: Check if the image file exists and is accessible
        if not os.path.exists(image_path):
            print(f"Image not found: {image_path}")
            continue
        
        # Preprocess the image
        image = Image.open(image_path).convert('RGB')
        image = transform(image).unsqueeze(0)  # Add batch dimension

        # Debugging: Confirm image processing
        print(f"Processing image: {os.path.basename(image_path)}")

        # Make predictions
        with torch.no_grad():
            logits = model(image)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()  # Get predicted probabilities

        # Debugging: Print probabilities
        print(f"Predicted probabilities for {os.path.basename(image_path)}: {probs[:5]}...")  # Print first 5 values for brevity

        # Apply threshold to determine presence of each label
        predicted_labels = (probs >= threshold).astype(int)

        # Create a dictionary to store the predicted labels for each image
        image_predictions = {label_cols[i]: predicted_labels[i] for i in range(len(label_cols))}

        # Check if the generated image contains the information from the prompt
        prompt_conditions = {
            "Disease (Consolidation)": image_predictions['Consolidation'] == 1,
            "Age Group (50-70 years)": image_predictions['AGE_GROUP_AGE_51_70'] == 1,
            "Gender (Male)": image_predictions['GENDER_Male'] == 1,
            "Race (White)": image_predictions['PRIMARY_RACE_White'] == 1
        }

        # Print the predictions for each condition in the prompt
        print(f"Predictions for image: {os.path.basename(image_path)}")
        for condition, result in prompt_conditions.items():
            print(f"{condition}: {'Present' if result else 'Absent'}")
        print("-" * 50)

    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        continue  # Skip the problematic image and continue with the next


/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Model weights loaded successfully.
Model loaded and ready for inference!
Found 15 images.
Processing image: generated_xray_12.png
Predicted probabilities for generated_xray_12.png: [8.2446855e-01 2.0540778e-03 6.5912068e-04 1.7340431e-02 1.0082194e-02]...
Predictions for image: generated_xray_12.png
Disease (Consolidation): Absent
Age Group (50-70 years): Absent
Gender (Male): Present
Race (White): Absent
--------------------------------------------------
Processing image: generated_xray_4.png
Predicted probabilities for generated_xray_4.png: [9.3293333e-01 2.2230741e-02 4.9457472e-04 3.8582527e-03 1.9223448e-03]...
Predictions for image: generated_xray_4.png
Disease (Consolidation): Absent
Age Group (50-70 years): Absent
Gender (Male): Present
Race (White): Absent
--------------------------------------------------
Processing image: generated_xray_6.png
Predicted probabilities for generated_xray_6.png: [9.9674535e-01 1.1426008e-03 6.5836781e-03 5.1477575e-04 5.1209965e-05]...
Predictio